# Selective facts — corrected scope, existing weights (Google Colab version)

Colab port of `notebooks/kaggle/selective_facts_scopefix/selective_facts_scopefix_eval.ipynb`.
Same **inference-only** parser evaluation: it reuses the selected full epoch-2 1.5B adapter, trains nothing and runs no Stage 2 systems.

**What is identical to the Kaggle notebook:** the hash-verified source bundle (manifest `21063deacc43…`), the adapter verification, the scope-fix preflight (12 changed / 88 unchanged prompts), the two-task smoke, the 100-case evaluation and `comparison.json`.

**What differs (Colab only):**
- Inputs are read from a Google Drive folder instead of `/kaggle/input`; outputs are copied back to that folder.
- Kaggle's check for exactly `torch 2.11.0+cu130` on **two** T4 GPUs is replaced by a check for **one** CUDA GPU with at least 12 GB. The evaluation only ever uses one GPU (`CUDA_VISIBLE_DEVICES=0`), as on Kaggle.
- The torch build is whatever Colab provides (recorded in `colab_environment.json`). Greedy FP16 outputs can differ slightly from a Kaggle run because of the different torch/CUDA build.

## Before running
1. Put **both** files in one Google Drive folder, by default `MyDrive/igt_scopefix/`:
   - `selective_facts_scopefix_source.zip`
   - `selective_facts_1_5b.zip` (the complete training output, not the smoke ZIP)
2. **Runtime → Change runtime type → T4 GPU** (L4 or A100 also work).
3. **Runtime → Run all**. Approve the Google Drive access prompt.

The full run takes roughly 1–2 hours on a T4; keep the tab open. At the end, `selective_facts_scopefix_eval.zip`, the logs and `colab_environment.json` are copied to the Drive folder.


In [1]:
# ---- Settings: change DRIVE_FOLDER if you put the two ZIPs somewhere else ----
DRIVE_FOLDER = '/content/drive/MyDrive/igt_scopefix'
MIN_GPU_GB = 12

import datetime, hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

INPUT = Path(DRIVE_FOLDER)
assert INPUT.is_dir(), f'Create {INPUT} in Google Drive and put the two ZIP files in it.'
for name in ['selective_facts_scopefix_source.zip', 'selective_facts_1_5b.zip']:
    found = list(INPUT.rglob(name))
    print(('found   ' if found else 'MISSING ') + name, *found)
RUN_ID = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
WORK = Path('/content/scopefix_work')          # local disk is much faster than Drive
WORK.mkdir(parents=True, exist_ok=True)
RUN_DIR = WORK / ('run_' + RUN_ID)             # fresh output folder for every run, so reruns never collide
RUN_DIR.mkdir()
DRIVE_RESULTS = INPUT / ('results_' + RUN_ID)
print('Run folder:', RUN_DIR, '\nResults will be copied to:', DRIVE_RESULTS)
subprocess.run(['nvidia-smi'], check=False)

Mounted at /content/drive
found   selective_facts_scopefix_source.zip /content/drive/MyDrive/igt_scopefix/selective_facts_scopefix_source.zip
found   selective_facts_1_5b.zip /content/drive/MyDrive/igt_scopefix/selective_facts_1_5b.zip
Run folder: /content/scopefix_work/run_20261114_160333 
Results will be copied to: /content/drive/MyDrive/igt_scopefix/results_20261114_160333


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
# ---- Verify and unpack the source bundle (same checks as the Kaggle notebook) ----
MANIFEST_SHA = '21063deacc439e196c07bad84cf4b19ba25b99d2cf8f85a49587d457e7bb0e9b'
MANIFEST_NAME = 'selective_facts_scopefix_manifest.json'
ROOT = WORK / ('selective_facts_scopefix_source_' + MANIFEST_SHA[:12])

def write_source(manifest_bytes, read_source):
    assert hashlib.sha256(manifest_bytes).hexdigest() == MANIFEST_SHA, 'Wrong source/data bundle version'
    manifest = json.loads(manifest_bytes)
    assert manifest['experiment'] == 'selective-facts-scopefix1'
    for name, expected in manifest['sha256'].items():
        relative = Path(name)
        assert not relative.is_absolute() and '..' not in relative.parts, 'Invalid source path'
        content = read_source(name)
        assert hashlib.sha256(content).hexdigest() == expected, f'Changed source/data: {name}'
        path = ROOT / relative; path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists(): assert path.read_bytes() == content, f'Changed working copy: {name}'
        else: path.write_bytes(content)
    print('Verified source and data:', len(manifest['sha256']), 'files', MANIFEST_SHA)

manifest_paths = [p for p in INPUT.rglob(MANIFEST_NAME) if hashlib.sha256(p.read_bytes()).hexdigest() == MANIFEST_SHA]
if manifest_paths:                                    # the ZIP was unpacked in Drive
    manifest_path = manifest_paths[0]
    write_source(manifest_path.read_bytes(), lambda name: (manifest_path.parent / name).read_bytes())
else:                                                 # the ZIP itself is in Drive (normal case)
    found = False
    for archive in INPUT.rglob('selective_facts_scopefix_source.zip'):
        with zipfile.ZipFile(archive) as z:
            assert len(z.namelist()) == len(set(z.namelist())), 'Duplicate source ZIP members'
            name = 'selective_facts_scopefix_source/' + MANIFEST_NAME
            if name not in z.namelist(): continue
            manifest_bytes = z.read(name)
            if hashlib.sha256(manifest_bytes).hexdigest() != MANIFEST_SHA: continue
            write_source(manifest_bytes, lambda relative: z.read('selective_facts_scopefix_source/' + relative))
            found = True; break
    assert found, f'Put selective_facts_scopefix_source.zip in {INPUT}.'

PACKAGES = WORK / 'selective_facts_scopefix_packages'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--target', str(PACKAGES),
                '-r', str(ROOT / 'requirements.txt')], check=True)
ENV = dict(os.environ, PYTHONPATH=os.pathsep.join(map(str, [ROOT, ROOT / 'src', PACKAGES])),
           PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false')

Verified source and data: 110 files 21063deacc439e196c07bad84cf4b19ba25b99d2cf8f85a49587d457e7bb0e9b


In [3]:
# ---- Inference packages and preflight checks ----
# Same pinned inference packages as the Kaggle notebook. torch is not pinned: Colab's own CUDA build is used.
# pip may print dependency-conflict warnings for unrelated preinstalled Colab packages; they do not affect this run.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.20.2', 'peft==0.21.2',
                'bitsandbytes==0.50.2', 'accelerate==1.15.0', 'tokenizers==0.23.3', 'safetensors==0.8.0',
                'huggingface-hub==1.33.0'], check=True)
subprocess.run([sys.executable, '-m', 'eval.stage2_independent_v2.environment', '--prepare'], cwd=ROOT, env=ENV, check=True)

# Colab replacement for Kaggle's "torch 2.11.0+cu130 on T4 x2" assertion: one CUDA GPU with enough memory.
GPU_CHECK = f"""
import json, torch, transformers, peft, bitsandbytes
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
p = torch.cuda.get_device_properties(0)
gb = p.total_memory / 2**30
assert gb >= {MIN_GPU_GB}, f'GPU has only {{gb:.1f}} GB'
info = dict(gpu=p.name, gpu_memory_gb=round(gb, 1), gpu_count=torch.cuda.device_count(), torch=torch.__version__,
            cuda=torch.version.cuda, transformers=transformers.__version__, peft=peft.__version__,
            bitsandbytes=bitsandbytes.__version__, kaggle_reference_torch='2.11.0+cu130')
print(json.dumps(info))
open(r'{RUN_DIR / "colab_environment.json"}', 'w').write(json.dumps(info, indent=2))
"""
subprocess.run([sys.executable, '-c', GPU_CHECK], cwd=ROOT, env=dict(ENV, CUDA_VISIBLE_DEVICES='0'), check=True)
print('GPU preflight passed')

# Unchanged checks from the Kaggle notebook (inference uses one visible GPU).
subprocess.run([sys.executable, '-m', 'finetune.selective_facts.attention'], cwd=ROOT,
               env=dict(ENV, CUDA_VISIBLE_DEVICES='0'), check=True)
subprocess.run([sys.executable, '-m', 'finetune.selective_facts_scopefix.preflight'], cwd=ROOT, env=ENV, check=True)

# Adapter discovery: same verification, but searching the Drive folder instead of /kaggle/input.
ADAPTER_PATH_FILE = RUN_DIR / 'scopefix_adapter_path.txt'
subprocess.run([sys.executable, '-m', 'finetune.selective_facts_scopefix.adapter', '--input', str(INPUT),
                '--work', str(WORK), '--out', str(ADAPTER_PATH_FILE)], cwd=ROOT, env=ENV, check=True)
ADAPTER = Path(ADAPTER_PATH_FILE.read_text())
print('Existing epoch-2 adapter:', ADAPTER)

GPU preflight passed
Existing epoch-2 adapter: /content/scopefix_work/scopefix_existing_adapter


In [4]:
# ---- Runner: streams the log and always copies results (also partial ones) to Drive ----
def save_to_drive():
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    for p in RUN_DIR.iterdir():
        if p.is_file() and p.suffix in ('.zip', '.log', '.json', '.txt'):
            shutil.copy2(p, DRIVE_RESULTS / p.name)
    print('Copied to Drive:', sorted(x.name for x in DRIVE_RESULTS.iterdir()))

def execute(smoke):
    out = RUN_DIR / ('selective_facts_scopefix_smoke' if smoke else 'selective_facts_scopefix_eval')
    assert not out.exists(), 'Existing results are never overwritten; rerun the first cell for a new run folder.'
    command = [sys.executable, '-m', 'finetune.selective_facts_scopefix.evaluate', '--adapter', str(ADAPTER),
               '--out', str(out), '--packages', str(PACKAGES)]
    if smoke: command += ['--smoke']
    log = RUN_DIR / (out.name + '.log')
    try:
        with log.open('w') as f:
            process = subprocess.Popen(command, cwd=ROOT, env=dict(ENV, CUDA_VISIBLE_DEVICES='0'),
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end=''); f.write(line); f.flush()
            code = process.wait()
    finally:
        save_to_drive()
    assert code == 0, f'Job failed. The log and any partial ZIP are in {DRIVE_RESULTS}.'
    assert json.loads((out / 'completion.json').read_text())['complete']
    return out

SMOKE = execute(True)
print('Two-task inference smoke completed. The next cell evaluates all 100 development cases.')


Loading weights: 100%|██████████| 338/338 [00:03<00:00, 108.77it/s]
DEV 1/2 warehouse_fill_0: ok
DEV 2/2 warehouse_dedup_0: ok
Copied to Drive: ['colab_environment.json', 'scopefix_adapter_path.txt', 'selective_facts_scopefix_smoke.log', 'selective_facts_scopefix_smoke.zip']
Two-task inference smoke completed. The next cell evaluates all 100 development cases.


In [5]:
# ---- Full evaluation: all 100 open-development cases ----
OUT = execute(False)
comparison = json.loads((OUT / 'comparison.json').read_text())
print(json.dumps(comparison, indent=2))
print('\nDone. selective_facts_scopefix_eval.zip is in', DRIVE_RESULTS)
print('Also download this executed notebook: File -> Download -> Download .ipynb. No training was run.')


Loading weights: 100%|██████████| 338/338 [00:04<00:00, 83.08it/s] 
DEV 1/100 warehouse_filter_0: invalid_contract
DEV 2/100 warehouse_filter_1: ok
DEV 3/100 warehouse_filter_2: ok
DEV 4/100 warehouse_filter_3: ok
DEV 5/100 warehouse_dedup_0: ok
DEV 6/100 warehouse_dedup_1: ok
DEV 7/100 warehouse_dedup_2: ok
DEV 8/100 warehouse_dedup_3: ok
DEV 9/100 warehouse_fill_0: ok
DEV 10/100 warehouse_fill_1: ok
DEV 11/100 warehouse_fill_2: ok
DEV 12/100 warehouse_fill_3: ok
DEV 13/100 warehouse_convert_0: ok
DEV 14/100 warehouse_convert_1: invalid_contract
DEV 15/100 warehouse_convert_2: ok
DEV 16/100 warehouse_convert_3: ok
DEV 17/100 warehouse_edit_0: ok
DEV 18/100 warehouse_edit_1: ok
DEV 19/100 warehouse_edit_2: ok
DEV 20/100 warehouse_edit_3: ok
DEV 21/100 solar_filter_0: ok
DEV 22/100 solar_filter_1: ok
DEV 23/100 solar_filter_2: ok
DEV 24/100 solar_filter_3: ok
DEV 25/100 solar_dedup_0: ok
DEV 26/100 solar_dedup_1: ok
DEV 27/100 solar_dedup_2: ok
DEV 28/100 solar_dedup_3: ok
DEV 29/100 s